In [1]:
#Import packages
from pdbfixer import PDBFixer
from openmm.app import PDBFile

In [2]:

#Load the PBD file
fixer = PDBFixer(filename='/home/users/ys472/ying_Project/kinase_inhibitor_design/structures/mark2_caga.pdb')

In [3]:
# Add missing residues and atoms
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(pH=7.0)

In [4]:
# Save cleaned PDB
with open('mark2.cleaned.pdb', 'w') as f:
    PDBFile.writeFile(fixer.topology, fixer.positions, f)

In [6]:
# Removes waters, non-standard heteroatoms, and any unwanted molecules.
from Bio.PDB import PDBParser, PDBIO, Select

class KeepChainAandB(Select):
    def accept_chain(self, chain):
        return chain.id in ['A', 'B']  # Keep only chains A and B

parser = PDBParser(QUIET=True)
structure = parser.get_structure('protein', 'mark2.cleaned.pdb')

io = PDBIO()
io.set_structure(structure)
io.save('mark2.cleaned.filtered.pdb', select=KeepChainAandB())



In [7]:
for model in structure:
    for chain in model:
        for residue in chain:
            res_id = f"{chain.id}{residue.id[1]}"
            print(res_id, residue.resname)

A1 HIS
A2 ILE
A3 GLY
A4 ASN
A5 TYR
A6 ARG
A7 LEU
A8 LEU
A9 LYS
A10 THR
A11 ILE
A12 GLY
A13 ALA
A14 LYS
A15 VAL
A16 LYS
A17 LEU
A18 ALA
A19 ARG
A20 HIS
A21 ILE
A22 LEU
A23 THR
A24 GLY
A25 LYS
A26 GLU
A27 VAL
A28 ALA
A29 VAL
A30 LYS
A31 ILE
A32 ILE
A33 ASP
A34 LYS
A35 THR
A36 GLN
A37 LEU
A38 ASN
A39 SER
A40 SER
A41 SER
A42 LEU
A43 GLN
A44 LYS
A45 LEU
A46 PHE
A47 ARG
A48 GLU
A49 VAL
A50 ARG
A51 ILE
A52 MET
A53 LYS
A54 VAL
A55 LEU
A56 ASN
A57 HIS
A58 PRO
A59 ASN
A60 ILE
A61 VAL
A62 LYS
A63 LEU
A64 PHE
A65 GLU
A66 VAL
A67 ILE
A68 GLU
A69 THR
A70 GLU
A71 LYS
A72 THR
A73 LEU
A74 TYR
A75 LEU
A76 VAL
A77 MET
A78 GLU
A79 TYR
A80 ALA
A81 SER
A82 GLY
A83 GLY
A84 GLU
A85 VAL
A86 PHE
A87 ASP
A88 TYR
A89 LEU
A90 VAL
A91 ALA
A92 HIS
A93 GLY
A94 TRP
A95 MET
A96 LYS
A97 GLU
A98 LYS
A99 GLU
A100 ALA
A101 ARG
A102 ALA
A103 LYS
A104 PHE
A105 ARG
A106 GLN
A107 ILE
A108 VAL
A109 SER
A110 ALA
A111 VAL
A112 GLN
A113 TYR
A114 CYS
A115 HIS
A116 GLN
A117 LYS
A118 PHE
A119 ILE
A120 VAL
A121 HIS
A122 ARG
A123 ASP
A

In [8]:
#Script for checking if the pdb satisfies osprey K* requirement
from Bio.PDB import PDBParser
from collections import defaultdict

pdb_path = "pkn2.cleaned.filtered.pdb"
parser = PDBParser(QUIET=True)
structure = parser.get_structure("model", pdb_path)

# Track issues
missing_backbone = []
duplicate_atoms = defaultdict(list)
seen_atoms = set()

for model in structure:
    for chain in model:
        for residue in chain:
            res_id = f"{chain.id}{residue.id[1]}"
            atom_names = set()
            for atom in residue:
                key = (chain.id, residue.id[1], atom.name)
                if key in seen_atoms:
                    duplicate_atoms[res_id].append(atom.name)
                else:
                    seen_atoms.add(key)
                    atom_names.add(atom.name)
            # Check for missing backbone atoms
            required_backbone = {'N', 'CA', 'C', 'O'}
            if not required_backbone.issubset(atom_names):
                missing_backbone.append(res_id)

# Report
print("✅ Basic structure read successful.")
if duplicate_atoms:
    print("⚠️ Duplicate atoms found:")
    for res, atoms in duplicate_atoms.items():
        print(f"  {res}: {atoms}")
else:
    print("✅ No duplicate atoms.")

if missing_backbone:
    print("⚠️ Residues missing backbone atoms:")
    print(missing_backbone)
else:
    print("✅ All residues have full backbone.")

# Check for alternate conformers or insertion codes
altloc_issues = [
    (residue.id, residue.get_resname())
    for model in structure
    for chain in model
    for residue in chain
    if residue.id[2] != ' '  # insertion code or altloc
]

if altloc_issues:
    print("⚠️ Residues with alternate location or insertion code:")
    for rid, name in altloc_issues:
        print(f"  {rid}: {name}")
else:
    print("✅ No altLoc or insertion code issues.")


✅ Basic structure read successful.
✅ No duplicate atoms.
✅ All residues have full backbone.
✅ No altLoc or insertion code issues.
